# News Articles: Llama-3-8B Coherence Decay

Genre comparison: do news articles show the same ~0.78 exponent as Wikipedia?
Key test: does English news land in the cluster or steep like English Wiki?

In [ ]:
!pip install -q -U bitsandbytes>=0.46.1 accelerate

import numpy as np
import json, math, os, torch, gc
from scipy import stats
from pathlib import Path
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

from google.colab import drive
drive.mount('/content/drive')

NEWS_DIR = Path('/content/drive/MyDrive/LRTIA/Data/xlsum_news')
BASE_DIR = Path('/content/drive/MyDrive/LRTIA/Results/News_finegrain')
BASE_DIR.mkdir(parents=True, exist_ok=True)

MAX_CONTEXT = 100
TARGET_LEN = 30
TARGET_FRACTIONS = [0.25, 0.50, 0.75]
MIN_CONTEXT_BEFORE_TARGET = MAX_CONTEXT + 10
RANDOM_SEED = 42
common_x = np.arange(1, MAX_CONTEXT + 1)
bin_edges = [1, 2, 3, 4, 5, 7, 10, 15, 20, 30, 50, 75, 100]

print('Setup done')
print('News files:', [f.name for f in sorted(NEWS_DIR.glob('*.jsonl'))])

In [ ]:
# === Load Llama ===
device = 'cuda'
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

tokenizer = AutoTokenizer.from_pretrained('unsloth/Meta-Llama-3.1-8B')
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    'unsloth/Meta-Llama-3.1-8B',
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                                           bnb_4bit_compute_dtype=torch.float16),
    device_map='auto'
)
model.eval()
print('Model loaded')

In [ ]:
# === Functions ===
@torch.no_grad()
def compute_ppl(token_ids, target_start, target_end):
    if target_start >= target_end - 1: return float('inf')
    input_ids = torch.tensor([token_ids], device=model.device)
    outputs = model(input_ids)
    logits = outputs.logits[0]
    total_loss = 0.0
    count = 0
    for i in range(target_start, target_end - 1):
        log_probs = torch.log_softmax(logits[i], dim=-1)
        total_loss += -log_probs[token_ids[i + 1]].item()
        count += 1
    del outputs, logits
    torch.cuda.empty_cache()
    return math.exp(total_loss / count) if count > 0 else float('inf')

def process_document(doc, rng_shuf):
    full_ids = tokenizer.encode(doc['text'], add_special_tokens=False)
    n = len(full_ids)
    intact_curves, shuffled_curves = [], []
    for frac in TARGET_FRACTIONS:
        target_start = int(n * frac)
        target_end = min(target_start + TARGET_LEN, n)
        if target_start < MIN_CONTEXT_BEFORE_TARGET or target_end - target_start < 5: continue
        target_ids = full_ids[target_start:target_end]
        context_pool = list(full_ids[:target_start])
        max_ctx = min(MAX_CONTEXT, len(context_pool))
        if max_ctx < 10: continue
        ppls, ctx_lengths = [], []
        for ctx_len in range(1, max_ctx + 1):
            chunk = context_pool[-ctx_len:] + target_ids
            ppl = compute_ppl(chunk, ctx_len, ctx_len + len(target_ids))
            if not math.isinf(ppl): ppls.append(ppl); ctx_lengths.append(ctx_len)
        if len(ppls) >= 10:
            intact_curves.append({'ctx_lengths': ctx_lengths, 'ppls': ppls,
                                  'doc_id': doc.get('doc_id',''), 'target_frac': frac})
        context_shuf = list(context_pool); rng_shuf.shuffle(context_shuf)
        ppls_s, ctx_s = [], []
        for ctx_len in range(1, max_ctx + 1):
            chunk = context_shuf[-ctx_len:] + target_ids
            ppl = compute_ppl(chunk, ctx_len, ctx_len + len(target_ids))
            if not math.isinf(ppl): ppls_s.append(ppl); ctx_s.append(ctx_len)
        if len(ppls_s) >= 10:
            shuffled_curves.append({'ctx_lengths': ctx_s, 'ppls': ppls_s,
                                    'doc_id': doc.get('doc_id',''), 'target_frac': frac})
    return intact_curves, shuffled_curves

def compute_raw_ppl_curve(curves):
    all_ppl = []
    for curve in curves:
        ctx = np.array(curve['ctx_lengths'])
        ppl = np.array(curve['ppls'])
        interp = np.interp(common_x, ctx, ppl, left=np.nan, right=np.nan)
        all_ppl.append(interp)
    return np.nanmean(np.array(all_ppl), axis=0)

def fit_power_law(marg):
    bm, bc = [], []
    for i in range(len(bin_edges) - 1):
        lo, hi = bin_edges[i], bin_edges[i+1]
        vals = marg[lo-1:hi-1]
        vals = vals[~np.isnan(vals)]
        if len(vals) > 0 and np.mean(vals) > 0:
            bm.append(np.mean(vals)); bc.append((lo + hi) / 2)
    if len(bm) >= 4:
        slope, intercept, r, p, _ = stats.linregress(np.log(bc), np.log(bm))
        return slope, r, p
    return None

print('Functions defined')

In [ ]:
# === Run news articles ===
# Wiki reference exponents (Llama)
WIKI_REF = {'en': -1.253, 'fr': -0.695, 'tr': -0.879, 'zh': -0.768}

for lang, name in [('en','English'), ('fr','French'), ('tr','Turkish'), ('zh','Chinese')]:
    print(f'\n{"="*50}')
    intact_path = BASE_DIR / f'llama_{lang}_news_intact.json'
    shuffled_path = BASE_DIR / f'llama_{lang}_news_shuffled.json'

    if intact_path.exists() and shuffled_path.exists():
        with open(intact_path) as f: intact = json.load(f)
        with open(shuffled_path) as f: shuffled = json.load(f)
        print(f'{name} news: cached ({len(intact)} curves)')
    else:
        news_path = NEWS_DIR / f'{lang}_news.jsonl'
        if not news_path.exists():
            print(f'{name} news: NO DATA'); continue
        corpus = []
        with open(news_path) as f:
            for line in f: corpus.append(json.loads(line))
        print(f'{name} news: {len(corpus)} articles')
        intact, shuffled = [], []
        rng_shuf = np.random.RandomState(RANDOM_SEED + 99)
        for doc in tqdm(corpus, desc=f'{name} news'):
            i, s = process_document(doc, rng_shuf)
            intact.extend(i); shuffled.extend(s)
        with open(intact_path, 'w') as f: json.dump(intact, f)
        with open(shuffled_path, 'w') as f: json.dump(shuffled, f)

    if len(intact) >= 5 and len(shuffled) >= 5:
        ip = compute_raw_ppl_curve(intact)
        sp = compute_raw_ppl_curve(shuffled)
        corr = -np.diff(ip) - (-np.diff(sp))
        fit = fit_power_law(corr)
        if fit:
            wiki_ref = WIKI_REF.get(lang, 0)
            print(f'  >> {name} NEWS:  α = {fit[0]:.3f} (r={fit[1]:.3f}, n={len(intact)})')
            print(f'     {name} WIKI:  α = {wiki_ref:.3f} (reference)')
            print(f'     Δ(news-wiki) = {fit[0]-wiki_ref:.3f}')

## Key question

If English news lands near -0.78 (like all other languages' Wiki),
then English Wikipedia is the outlier, not English as a language.

If English news is also steep (~-1.25), then something about English
text specifically decays faster in Llama's view.